# Week 12 — Word Embeddings & Semantic Similarity

**Theme:** NLP basics — representing words as vectors

A neural network can't read words directly — it needs numbers. The simplest
idea (assign each word a random ID) throws away all meaning: ID 47 and ID 48
could be totally unrelated words. **Word embeddings** fix this by learning a
vector for each word such that words used in *similar contexts* end up with
*similar vectors* — meaning becomes geometry.

We'll train `gensim`'s **Word2Vec** on a small hand-built corpus (real corpora
have millions of sentences; ours has a few thousand, built from templates —
enough to see the idea work, not enough to be great at it).

In [ ]:
!pip install -q gensim

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

random.seed(0)

## 1. Build a small corpus from templates

Real training data is millions of naturally occurring sentences. We fake a
tiny version of that idea: pick related word pairs across a few categories,
plug them into sentence templates, and repeat with variation so the model
sees each word in enough different contexts to learn something.

In [ ]:
pairs_gender_role = [
    ("man", "king"), ("woman", "queen"),
    ("man", "actor"), ("woman", "actress"),
    ("man", "waiter"), ("woman", "waitress"),
    ("boy", "prince"), ("girl", "princess"),
]
capitals = [
    ("france", "paris"), ("germany", "berlin"), ("japan", "tokyo"),
    ("italy", "rome"), ("spain", "madrid"), ("korea", "seoul"),
]
animals_baby = [
    ("dog", "puppy"), ("cat", "kitten"), ("cow", "calf"), ("lion", "cub"),
]

templates_role = [
    "the {a} who rules the land is called a {b}",
    "a {a} became a {b} after the ceremony",
    "every {a} in the story grows up to be a {b}",
    "the {b} is a {a} with a crown",
]
templates_capital = [
    "{cap} is the capital city of {country}",
    "if you visit {country} you should see {cap}",
    "{country} chose {cap} as its capital",
]
templates_animal = [
    "the {adult} has a baby called a {baby}",
    "a {baby} grows up to become an adult {adult}",
    "the {adult} and its {baby} played together",
]

sentences = []
for _ in range(80):
    for a, b in pairs_gender_role:
        sentences.append(random.choice(templates_role).format(a=a, b=b).split())
    for country, cap in capitals:
        sentences.append(random.choice(templates_capital).format(cap=cap, country=country).split())
    for adult, baby in animals_baby:
        sentences.append(random.choice(templates_animal).format(adult=adult, baby=baby).split())

print(f"Corpus size: {len(sentences)} sentences")
print("Example sentences:", sentences[0], "|", sentences[80], "|", sentences[-1])

## 2. Train Word2Vec

`vector_size` is how many numbers represent each word. `window` is how many
neighboring words count as "context." `sg=1` uses the skip-gram variant
(predict context words from the current word), which tends to work better on
small corpora like ours.

In [ ]:
model = Word2Vec(
    sentences,
    vector_size=30,
    window=4,
    min_count=1,
    sg=1,
    epochs=300,   # a real corpus needs far fewer passes -- ours is tiny, so we compensate
    seed=0,
    workers=1,
)
print("Vocabulary size:", len(model.wv))
print("Vector for 'king':", model.wv["king"][:6], "... (30 numbers total)")

## 3. Nearest neighbors: `most_similar`

Ask the model: which words have the closest vectors to a given word?

In [ ]:
for word in ["paris", "king", "puppy"]:
    print(f"Most similar to '{word}':")
    for neighbor, score in model.wv.most_similar(word, topn=5):
        print(f"   {neighbor:12s} similarity={score:.3f}")
    print()

Notice `paris`'s neighbors are almost entirely *other capital cities* — the
model has learned "these words play the same role in sentences" purely from
which words tend to appear near each other, without ever being told what a
"capital" is.

## 4. Vector arithmetic: analogies

The famous Word2Vec party trick: `king - man + woman ≈ queen`. In vector
terms, `most_similar(positive=["king", "woman"], negative=["man"])` finds the
word whose vector is closest to `vector("king") - vector("man") + vector("woman")`.

In [ ]:
result = model.wv.most_similar(positive=["paris", "germany"], negative=["france"], topn=5)
print("paris - france + germany ≈", result)

In [ ]:
result = model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=5)
print("king - man + woman ≈", result)

The capital-city analogy usually lands correctly on `berlin` — the
country/capital relationship is expressed very consistently across our
templates, so the model captures it cleanly.

The gender analogy is messier: `queen` often shows up, but not always ranked
first, even though `queen` does appear near `king` in the "most similar"
results above. **This is normal, not a bug** — real Word2Vec/GloVe models
trained on billions of words show the same kind of inconsistency on gender
analogies. Embeddings capture *statistical* patterns in how words are used,
which only approximates human semantic relationships — and our tiny toy
corpus makes that approximation even rougher.

## 5. Visualize the embedding space (connects back to Week 4)

30 numbers per word can't be plotted directly — so, just like Week 4, we use
PCA to compress each word vector down to 2D.

In [ ]:
categories = {}
for a, b in pairs_gender_role:
    categories[a] = "gender/role"
    categories[b] = "gender/role"
for country, cap in capitals:
    categories[country] = "country"
    categories[cap] = "capital"
for adult, baby in animals_baby:
    categories[adult] = "animal (adult)"
    categories[baby] = "animal (baby)"

words = list(categories.keys())
vectors = np.array([model.wv[w] for w in words])
vectors_2d = PCA(n_components=2).fit_transform(vectors)

color_map = {
    "gender/role": "steelblue", "country": "darkorange",
    "capital": "seagreen", "animal (adult)": "indianred", "animal (baby)": "orchid",
}
colors = [color_map[categories[w]] for w in words]

plt.figure(figsize=(9, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], c=colors, s=80)
for i, w in enumerate(words):
    plt.annotate(w, (vectors_2d[i, 0], vectors_2d[i, 1]), fontsize=9)
handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=cat)
           for cat, c in color_map.items()]
plt.legend(handles=handles, loc="best")
plt.title("Word Embeddings in 2D (PCA)")
plt.show()

## Try it yourself

1. **Add your own category.** Pick 4-6 related word pairs (e.g. fruits, sports,
   colors) and template sentences, add them to the corpus, retrain, and check
   whether they form their own cluster in the PCA plot.
2. **Fewer epochs.** Retrain with `epochs=5` — do the `most_similar` results
   get noticeably worse? Why does a tiny corpus need many more passes than a
   huge real-world one?
3. **`window` size.** Try `window=1` vs. `window=8` — how does the neighbor
   size for `king` or `paris` change?
4. **A harder analogy.** Try `cat - kitten + puppy` (should be close to `dog`)
   — does it work better or worse than the gender analogy? Why might that be,
   given how the animal templates were written?

---
## 🎯 캡스톤: 강의평 키워드로 나에게 맞는 수업 찾기

6개 가상 강의에 대한 강의평을 템플릿으로 생성한 더미 말뭉치를 드립니다. 위에서 배운 것과 같은 방식으로 Word2Vec을 학습시켜서, "내가 원하는 수업 스타일"을 키워드 몇 개로 입력하면 가장 잘 맞는 강의를 추천해주는 미니 추천기를 직접 만들어보세요.

**확장 아이디어:** 실제 학교 강의평 게시판(에브리타임 등)의 후기 문장들을 모아 `review_sentences`를 바꾸면, 진짜 "강의평 기반 수강신청 추천기"가 됩니다.

In [ ]:
# 더미 강의평 말뭉치 생성 (실행만 하면 됩니다)
import random
rng2 = random.Random(31)

course_keywords = {
    "인공지능개론": ["어렵다", "과제많다", "실습위주", "흥미롭다", "코딩많다"],
    "문학의이해":   ["쉽다", "편안하다", "발표없다", "읽기많다", "여유롭다"],
    "경제원론":     ["어렵다", "암기많다", "이론적이다", "시험어렵다", "지루하다"],
    "디자인씽킹":   ["재미있다", "팀플많다", "창의적이다", "실습위주", "흥미롭다"],
    "체육과건강":   ["쉽다", "몸움직인다", "실습위주", "편안하다", "재미있다"],
    "심리학개론":   ["흥미롭다", "발표있다", "읽기많다", "이론적이다", "재미있다"],
}

templates = [
    "{course} 강의는 {kw1}.",
    "{course} 수강후기: {kw1}하고 {kw2}.",
    "이번 학기 {course}는 확실히 {kw1}.",
    "{course} 들어보니 {kw1}, 그리고 {kw2}.",
]

review_sentences = []
for course, keywords in course_keywords.items():
    for _ in range(100):
        t = rng2.choice(templates)
        kw1, kw2 = rng2.sample(keywords, 2)
        review_sentences.append(t.format(course=course, kw1=kw1, kw2=kw2).split())

print(f"강의평 문장 수: {len(review_sentences)}")
print("예시:", review_sentences[0], "|", review_sentences[300])

### 여러분의 과제

1. Week 12 본문과 같은 방식으로 `review_sentences`에 `Word2Vec`을 학습시키세요 (`sg=1`, 충분한 `epochs`).
2. 각 강의(`course_keywords`의 key)에 대해, 그 강의의 키워드 벡터들을 평균 내서 **"강의 프로필 벡터"**를 만드세요.
3. 아래 `my_keywords`에 여러분이 원하는 수업 스타일 키워드 2~3개를 입력하고, 그 키워드들의 평균 벡터("질의 벡터")를 구하세요.
4. 질의 벡터와 각 강의 프로필 벡터 사이의 코사인 유사도를 계산해서, **가장 잘 맞는 강의 순위**를 출력하세요. (힌트: `numpy`로 직접 코사인 유사도를 계산하거나, `model.wv.cosine_similarities()`를 활용하세요.)
5. (선택) Week 12 본문처럼 PCA로 모든 단어 벡터를 2D로 시각화해서, 강의 이름과 키워드들이 어떻게 모여 있는지 확인해보세요.

In [ ]:
# TODO 1: review_sentences로 Word2Vec을 학습시키세요.


# TODO 2: 각 강의의 "프로필 벡터"(키워드 벡터 평균)를 계산하세요.


# TODO 3-4: 나의 희망 키워드를 입력하고, 가장 잘 맞는 강의 순위를 코사인 유사도로 계산해보세요.
my_keywords = []  # 예: ["쉽다", "편안하다"]

# TODO 5 (선택): PCA로 전체 단어 벡터를 2D로 시각화해보세요.